In [ ]:
import os, gdown, zipfile

os.makedirs("/home/jovyan/data", exist_ok=True)
os.makedirs("/home/jovyan/data/rivers", exist_ok=True)
os.makedirs("/home/jovyan/data/study_area", exist_ok=True)

# ── File IDs ──────────────────────────────────────────────────
NC_ID    = "1tYIrP_39e2aj8OdWztyPiX-hrayF-ueP"
RIVER_ID = "1sQ9xfnfIrdtuZD0eji1p86pDeJbqKAN5"
STUDY_ID = "1l2-LmnWyVd0bJcCCXSf6CdaJdPUFS4eq"
# ──────────────────────────────────────────────────────────────

def _download(file_id, out_path, label):
    if not os.path.exists(out_path):
        print(f"⬇️  Downloading {label}...")
        gdown.download(
            f"https://drive.google.com/uc?id={file_id}",
            out_path, quiet=False, fuzzy=True)
        print(f"✓ {label} done")
    else:
        print(f"✓ {label} already present — skipping")

def _download_zip(file_id, zip_path, extract_dir, label):
    already = any(f.endswith(".shp") for f in os.listdir(extract_dir))
    if not already:
        _download(file_id, zip_path, label)
        print(f"  Extracting {label}...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        os.remove(zip_path)
        print(f"✓ {label} extracted")
    else:
        print(f"✓ {label} already extracted — skipping")

_download(
    NC_ID,
    "/home/jovyan/data/combined_doppio_gomofs_2km_20070101_20251231.nc",
    "NetCDF (~10 GB — this will take 20-40 min ☕)")

_download_zip(
    RIVER_ID,
    "/home/jovyan/data/rivers/rivers.zip",
    "/home/jovyan/data/rivers",
    "River shapefile")

_download_zip(
    STUDY_ID,
    "/home/jovyan/data/study_area/study.zip",
    "/home/jovyan/data/study_area",
    "Study area shapefile")

print("\n✓ All files ready — run the next cell to launch the dashboard")

⬇️  Downloading NetCDF (~10 GB — this will take 20-40 min ☕)...


Downloading...
From (original): https://drive.google.com/uc?id=1tYIrP_39e2aj8OdWztyPiX-hrayF-ueP
From (redirected): https://drive.google.com/uc?id=1tYIrP_39e2aj8OdWztyPiX-hrayF-ueP&confirm=t&uuid=39ec56d9-428f-4bc1-af6b-9f2cc5d719b4
To: /home/jovyan/data/combined_doppio_gomofs_2km_20070101_20251231.nc
 48%|████▊     | 5.50G/11.4G [12:25<11:42, 8.41MB/s]

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   OCEAN DASHBOARD v7  —  Scattermapbox compatible           ║
# ╚══════════════════════════════════════════════════════════════╝

import os, warnings
import gc
import dask
dask.config.set({"array.chunk-size": "32MiB"})  # limit chunk size for Binder
import numpy as np
import pandas as pd
import xarray as xr
import plotly.graph_objects as go
import plotly.colors as pc
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely.ops import unary_union

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════
# 0.  USER CONFIG
# ═══════════════════════════════════════════════════════════════

LAT_MIN, LAT_MAX = 35.0, 48.0
LON_MIN, LON_MAX = -76.0, -60.0
TIME_CHUNK = 10

def _find_shp(folder):
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith(".shp"):
                return os.path.join(root, f)
    raise FileNotFoundError(f"No .shp found in {folder}")

COMBINED_NC = "/home/jovyan/data/combined_doppio_gomofs_2km_20070101_20251231.nc"
RIVER_SHP   = _find_shp("/home/jovyan/data/rivers")
STUDY_SHP   = _find_shp("/home/jovyan/data/study_area")

print(f"NetCDF     : {COMBINED_NC}")
print(f"Rivers     : {RIVER_SHP}")
print(f"Study area : {STUDY_SHP}")

# ═══════════════════════════════════════════════════════════════
# 1.  DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════

import shapely, plotly
print(f"plotly version  : {plotly.__version__}")
print(f"shapely version : {shapely.__version__}")
print(f"NetCDF exists   : {os.path.exists(COMBINED_NC)}")
print(f"River SHP exists: {os.path.exists(RIVER_SHP)}")
print(f"Study SHP exists: {os.path.exists(STUDY_SHP)}")

# ═══════════════════════════════════════════════════════════════
# 2.  OPEN NETCDF LAZILY
# ═══════════════════════════════════════════════════════════════

print("\nOpening NetCDF (lazy)...")
ds_nc    = xr.open_dataset(COMBINED_NC, chunks={"time": TIME_CHUNK})
nc_times = pd.to_datetime(ds_nc.time.values)
nc_lat2d = ds_nc.lat.values
nc_lon2d = ds_nc.lon.values
print(f"  Grid: {nc_lat2d.shape}   Time steps: {len(nc_times)}")
print(f"  Variables: {list(ds_nc.data_vars)}")

# ═══════════════════════════════════════════════════════════════
# 3.  STUDY AREA + RIVERS (FIXED)
# ═══════════════════════════════════════════════════════════════

# Fix for missing .shx files
os.environ["SHAPE_RESTORE_SHX"] = "YES"

print("\nLoading study area...")
study = gpd.read_file(STUDY_SHP)

# Fix for missing .prj in study area
if study.crs is None:
    print("  ! Study area missing CRS, assuming EPSG:4326")
    study.set_crs("EPSG:4326", allow_override=True, inplace=True)
study = study.to_crs("EPSG:4326")
study_union = unary_union(study.geometry)

# Extract study area lines for plotting
study_lats, study_lons = [], []
for geom in study.geometry:
    if geom is None: continue
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for poly in polys:
        xs, ys = poly.exterior.xy
        study_lons += list(xs) + [None]
        study_lats += list(ys) + [None]

def make_dashed(lts, lns, step=2):
    dl, dn = [], []
    for i, (la, lo) in enumerate(zip(lts, lns)):
        if la is None:       dl.append(None); dn.append(None)
        elif i % step == 0:  dl.append(la);   dn.append(lo)
        else:                dl.append(None); dn.append(None)
    return dl, dn

study_lats_d, study_lons_d = make_dashed(study_lats, study_lons)

print("Loading rivers...")
rivers = gpd.read_file(RIVER_SHP, bbox=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX))

# Fix for missing .prj in rivers
if rivers.crs is None:
    print("  ! Rivers missing CRS, assuming EPSG:4326")
    rivers.set_crs("EPSG:4326", allow_override=True, inplace=True)

# Safety check for the ORD_FLOW column
if "ORD_FLOW" in rivers.columns:
    print("  Filtering rivers by flow order...")
    rivers = rivers[rivers["ORD_FLOW"] <= 4]
else:
    print("  Column 'ORD_FLOW' not found; showing all rivers in file.")

rivers = rivers.to_crs("EPSG:4326")

river_lats, river_lons = [], []
for geom in rivers.geometry:
    if geom is None: continue
    # Handle both LineString and MultiLineString
    lines = [geom] if geom.geom_type == "LineString" else list(geom.geoms)
    for line in lines:
        xs, ys = line.xy
        river_lons += list(xs) + [None]
        river_lats += list(ys) + [None]

print("✓ All geometries loaded successfully.")

# ═══════════════════════════════════════════════════════════════
# 4.  SPATIAL MASK  — Shapely 1.x and 2.x compatible
# ═══════════════════════════════════════════════════════════════

nc_lat_f = nc_lat2d.ravel()
nc_lon_f = nc_lon2d.ravel()

shapely_major = int(shapely.__version__.split(".")[0])
if shapely_major >= 2:
    from shapely import contains_xy
    in_study = contains_xy(study_union, nc_lon_f, nc_lat_f)
else:
    from shapely.vectorized import contains
    in_study = contains(study_union, nc_lon_f, nc_lat_f)

SPATIAL_MASK = (
    (nc_lat_f >= LAT_MIN) & (nc_lat_f <= LAT_MAX) &
    (nc_lon_f >= LON_MIN) & (nc_lon_f <= LON_MAX) &
    in_study
)
print(f"  Spatial mask: {SPATIAL_MASK.sum():,} valid pixels")

# ═══════════════════════════════════════════════════════════════
# 5.  ON-DEMAND STATS CACHE
# ═══════════════════════════════════════════════════════════════

_stats_cache = {}

def _compute_stats(var):
    if var in _stats_cache:
        return _stats_cache[var]
    import gc
    print(f"  Computing stats for {var} (memory-efficient chunked)...")

    da = ds_nc[var].where(np.abs(ds_nc[var]) < 1e6)

    # Compute mean and std separately and delete intermediates immediately
    print(f"    computing mean...")
    mn = da.mean(dim="time").compute().values.astype(np.float32)
    gc.collect()

    print(f"    computing std...")
    sd = da.std(dim="time").compute().values.astype(np.float32)
    gc.collect()

    with np.errstate(invalid="ignore", divide="ignore"):
        cv = np.where(np.abs(mn) > 0.001, sd / np.abs(mn) * 100, np.nan)

    del sd
    gc.collect()

    result = {"mean": mn, "cv": cv, "presence": None}

    if var in ("gomofs_u", "gomofs_v"):
        print(f"    computing presence...")
        present = ds_nc["gomofs_u"].where(np.abs(ds_nc["gomofs_u"]) < 1e6).notnull()
        result["presence"] = present.mean(dim="time").compute().values.astype(np.float32) * 100
        del present
        gc.collect()

    _stats_cache[var] = result
    print(f"  ✓ {var} stats ready")
    return result

def get_flat(var, metric):
    stats = _compute_stats(var)
    flat  = stats[metric].ravel()
    valid = np.isfinite(flat) & SPATIAL_MASK
    return flat, valid

# ═══════════════════════════════════════════════════════════════
# 6.  MAP CONFIG
# ═══════════════════════════════════════════════════════════════

COLORSCALE = {
    ("doppio_temp",  "mean"):     "RdYlBu_r",
    ("doppio_temp",  "cv"):       "YlOrRd",
    ("doppio_salt",  "mean"):     "Viridis",
    ("doppio_salt",  "cv"):       "YlOrRd",
    ("doppio_ubar",  "mean"):     "RdBu_r",
    ("doppio_ubar",  "cv"):       "YlOrRd",
    ("doppio_vbar",  "mean"):     "RdBu_r",
    ("doppio_vbar",  "cv"):       "YlOrRd",
    ("gomofs_u",     "mean"):     "RdBu_r",
    ("gomofs_u",     "cv"):       "YlOrRd",
    ("gomofs_u",     "presence"): "RdYlGn",
    ("gomofs_v",     "mean"):     "RdBu_r",
    ("gomofs_v",     "cv"):       "YlOrRd",
    ("gomofs_v",     "presence"): "RdYlGn",
}
MAP_LABEL = {
    ("doppio_temp",  "mean"):     "Mean Temp (°C)",
    ("doppio_temp",  "cv"):       "Temp CV (%)",
    ("doppio_salt",  "mean"):     "Mean Salinity (psu)",
    ("doppio_salt",  "cv"):       "Salt CV (%)",
    ("doppio_ubar",  "mean"):     "DOPPIO Ubar — Mean (m/s)",
    ("doppio_ubar",  "cv"):       "DOPPIO Ubar — CV (%)",
    ("doppio_vbar",  "mean"):     "DOPPIO Vbar — Mean (m/s)",
    ("doppio_vbar",  "cv"):       "DOPPIO Vbar — CV (%)",
    ("gomofs_u",     "mean"):     "GOMOFS U — Mean (m/s)",
    ("gomofs_u",     "cv"):       "GOMOFS U — CV (%)",
    ("gomofs_u",     "presence"): "GOMOFS Data Presence (%)",
    ("gomofs_v",     "mean"):     "GOMOFS V — Mean (m/s)",
    ("gomofs_v",     "cv"):       "GOMOFS V — CV (%)",
    ("gomofs_v",     "presence"): "GOMOFS Data Presence (%)",
}
METRIC_OPTIONS = {
    "doppio_temp":  [("Mean", "mean"), ("CV (%)", "cv")],
    "doppio_salt":  [("Mean", "mean"), ("CV (%)", "cv")],
    "doppio_ubar":  [("Mean", "mean"), ("CV (%)", "cv")],
    "doppio_vbar":  [("Mean", "mean"), ("CV (%)", "cv")],
    "gomofs_u":     [("Mean", "mean"), ("CV (%)", "cv"), ("Data Presence (%)", "presence")],
    "gomofs_v":     [("Mean", "mean"), ("CV (%)", "cv"), ("Data Presence (%)", "presence")],
}
VAR_LABEL = {
    "doppio_temp":  "Temperature (°C)",
    "doppio_salt":  "Salinity (psu)",
    "doppio_ubar":  "DOPPIO Ubar (m/s)",
    "doppio_vbar":  "DOPPIO Vbar (m/s)",
    "gomofs_u":     "GOMOFS U (m/s)",
    "gomofs_v":     "GOMOFS V (m/s)",
}

# ═══════════════════════════════════════════════════════════════
# 7.  CLICK STATE  —  A = cool palette, B = warm palette
# ═══════════════════════════════════════════════════════════════

click_state  = {"A": None, "B": None}
next_slot    = ["A"]
MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
LOC_COLORS   = {"A": "#42a5f5", "B": "#ef5350"}   # marker pins: blue / red

# Year palettes — A gets cool (greens→blues→purples), B gets warm (reds→oranges→yellows)
def _year_palette(slot, n):
    if slot == "A":
        scales = ["#1b5e20","#2e7d32","#388e3c","#43a047","#00695c",
                  "#00796b","#00838f","#0277bd","#01579b","#283593",
                  "#4527a0","#6a1b9a","#7b1fa2","#8e24aa","#ab47bc"]
    else:
        scales = ["#b71c1c","#c62828","#d32f2f","#e53935","#e64a19",
                  "#bf360c","#e65100","#ef6c00","#f57c00","#ff8f00",
                  "#f9a825","#f57f17","#fdd835","#ffee58","#fff176"]
    return [scales[i % len(scales)] for i in range(n)]

# ═══════════════════════════════════════════════════════════════
# 8.  WIDGETS
# ═══════════════════════════════════════════════════════════════

var_dd = widgets.Dropdown(
    options=[("Temperature (°C)", "doppio_temp"),
             ("Salinity (psu)",   "doppio_salt"),
             ("DOPPIO Ubar (m/s)","doppio_ubar"),
             ("DOPPIO Vbar (m/s)","doppio_vbar"),
             ("GOMOFS U (m/s)",   "gomofs_u"),
             ("GOMOFS V (m/s)",   "gomofs_v")],
    value="doppio_temp", description="Variable:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="260px"))

map_dd = widgets.Dropdown(
    options=[("Mean", "mean"), ("CV (%)", "cv")],
    value="mean", description="Map metric:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="230px"))

nbr_dd = widgets.Dropdown(
    options=[("Single pixel", 0), ("±2 px", 2), ("±5 px", 5),
             ("±10 px", 10), ("±40 px", 40)],
    value=0, description="Neighborhood:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="230px"))

plot_btn  = widgets.Button(description="▶ Plot Time Series",
                           button_style="primary",
                           layout=widgets.Layout(width="170px"))
reset_btn = widgets.Button(description="✕ Reset Locations",
                           button_style="warning",
                           layout=widgets.Layout(width="160px"))

status_html  = widgets.HTML(
    "<span style='font-family:monospace;font-size:12px;color:#aaa;'>"
    "Click any point on the map to set Location A, then B</span>")
loading_html = widgets.HTML("")
out_ts       = widgets.Output()

# ═══════════════════════════════════════════════════════════════
# 9.  BUILD MAP FIGUREWIDGET  (Scattermapbox)
# ═══════════════════════════════════════════════════════════════

import plotly.basedatatypes as _pbd

if not getattr(_pbd.BaseFigure._perform_plotly_relayout, "_is_safe_patched", False):
    _orig_relayout = _pbd.BaseFigure._perform_plotly_relayout

    def _safe_relayout(self, relayout_data):
        filtered = {k: v for k, v in relayout_data.items()
                    if not k.endswith("._derived")}
        return _orig_relayout(self, filtered)

    _safe_relayout._is_safe_patched = True
    _pbd.BaseFigure._perform_plotly_relayout = _safe_relayout

def build_map_figure():
    fig = go.FigureWidget()
    fig.update_layout(
        mapbox=dict(
            style="white-bg",
            layers=[dict(
                below="traces", sourcetype="raster",
                source=["https://server.arcgisonline.com/ArcGIS/rest/services/"
                        "World_Imagery/MapServer/tile/{z}/{y}/{x}"])],
            center=dict(lat=(LAT_MIN + LAT_MAX) / 2,
                        lon=(LON_MIN + LON_MAX) / 2),
            zoom=5.0),
        title=dict(text="<b>Loading…</b>",
                   font=dict(size=14, color="#ddd"), x=0.5),
        height=620,
        margin=dict(t=45, b=5, l=5, r=5),
        paper_bgcolor="#0d1117",
        legend=dict(font=dict(color="#ccc", size=10)),
    )

    fig.add_trace(go.Scattermapbox(
        lat=[], lon=[], mode="markers",
        marker=dict(size=5, colorscale="RdYlBu_r",
                    colorbar=dict(
                        title=dict(text="", side="right"),
                        thickness=14,
                        title_font=dict(color="#ccc"),
                        tickfont=dict(color="#ccc")),
                    opacity=0.78),
        hovertemplate="Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<br>Val: %{marker.color:.3f}<extra></extra>",
        showlegend=False,
        name="data"))

    fig.add_trace(go.Scattermapbox(
        lat=river_lats, lon=river_lons, mode="lines",
        line=dict(color="#00cfff", width=1.5),
        hoverinfo="skip", showlegend=False))

    fig.add_trace(go.Scattermapbox(
        lat=study_lats_d, lon=study_lons_d, mode="lines",
        line=dict(color="white", width=2),
        hoverinfo="skip", showlegend=False))

    fig.add_trace(go.Scattermapbox(
        lat=[], lon=[], mode="markers+text",
        marker=dict(size=16, color=LOC_COLORS["A"]),
        text=[], textposition="top right",
        textfont=dict(color="white", size=13),
        showlegend=True, name="Location A"))

    fig.add_trace(go.Scattermapbox(
        lat=[], lon=[], mode="markers+text",
        marker=dict(size=16, color=LOC_COLORS["B"]),
        text=[], textposition="top right",
        textfont=dict(color="white", size=13),
        showlegend=True, name="Location B"))

    return fig

map_widget = build_map_figure()

# ═══════════════════════════════════════════════════════════════
# 10.  MAP DATA UPDATE
# ═══════════════════════════════════════════════════════════════

def update_map_data(var, metric):
    loading_html.value = (
        "<span style='font-family:monospace;font-size:11px;color:#f9a825;'>"
        f"⏳ Computing {var} {metric}…</span>")
    vals, valid = get_flat(var, metric)
    lbl  = MAP_LABEL.get((var, metric), f"{var} {metric}")
    cmap = COLORSCALE.get((var, metric), "Viridis")
    with map_widget.batch_update():
        map_widget.data[0].lat                        = nc_lat_f[valid]
        map_widget.data[0].lon                        = nc_lon_f[valid]
        map_widget.data[0].marker.color               = vals[valid]
        map_widget.data[0].marker.colorscale          = cmap
        map_widget.data[0].marker.colorbar.title.text = lbl
        map_widget.data[0].name                       = "data"
        map_widget.layout.title.text                  = (
            f"<b>{lbl}</b>  —  click a point to select location")
    loading_html.value = (
        "<span style='font-family:monospace;font-size:11px;color:#66bb6a;'>"
        f"✓ {lbl} ready</span>")

# ═══════════════════════════════════════════════════════════════
# 11-18. (OMITTED LOGIC — CONTINUING FROM YOUR EXISTING SCRIPT)
# ═══════════════════════════════════════════════════════════════

def update_markers():
    for slot, tidx in [("A", 3), ("B", 4)]:
        loc = click_state[slot]
        with map_widget.batch_update():
            if loc:
                map_widget.data[tidx].lat  = [loc["lat"]]
                map_widget.data[tidx].lon  = [loc["lon"]]
                map_widget.data[tidx].text = [slot]
            else:
                map_widget.data[tidx].lat  = []
                map_widget.data[tidx].lon  = []
                map_widget.data[tidx].text = []

def on_map_click(trace, points, selector):
    if not points.point_inds: return
    idx     = points.point_inds[0]
    new_lat = float(trace.lat[idx]); new_lon = float(trace.lon[idx])
    slot = next_slot[0]
    click_state[slot] = {"lat": new_lat, "lon": new_lon}
    next_slot[0] = "B" if slot == "A" else "A"
    update_markers()
    a, b = click_state["A"], click_state["B"]
    a_str = (f"<b style='color:{LOC_COLORS['A']}'>A: {a['lat']:.3f}, {a['lon']:.3f}</b>" if a else "A: —")
    b_str = (f"<b style='color:{LOC_COLORS['B']}'>B: {b['lat']:.3f}, {b['lon']:.3f}</b>" if b else "B: —")
    status_html.value = (f"<div style='font-family:monospace;background:#161b22;padding:5px;'>"
                         f"📍 {a_str} | {b_str} &nbsp; (next: {next_slot[0]})</div>")

map_widget.data[0].on_click(on_map_click)

def on_var_change(change):
    v = change["new"]; opts = METRIC_OPTIONS.get(v, [("Mean", "mean")])
    map_dd.options = opts; map_dd.value = opts[0][1]; update_map_data(v, map_dd.value)

def on_metric_change(change): update_map_data(var_dd.value, change["new"])
var_dd.observe(on_var_change, names="value"); map_dd.observe(on_metric_change, names="value")

def _nearest_nc(lat_c, lon_c):
    dist = np.abs(nc_lat2d - lat_c) + np.abs(nc_lon2d - lon_c)
    return np.unravel_index(np.nanargmin(dist), dist.shape)

def get_ts(var, lat_c, lon_c, nbr):
    yi, xi = _nearest_nc(lat_c, lon_c); H, W = nc_lat2d.shape
    if nbr == 0: ts = ds_nc[var].isel(y=yi, x=xi).values.astype(np.float64)
    else:
        r0, r1 = max(0, yi - nbr), min(H, yi + nbr + 1)
        c0, c1 = max(0, xi - nbr), min(W, xi + nbr + 1)
        ts = np.nanmean(ds_nc[var].isel(y=slice(r0, r1), x=slice(c0, c1)).values, axis=(1, 2)).astype(np.float64)
    ts[np.abs(ts) > 1e6] = np.nan; return nc_times, ts

def get_presence_ts(lat_c, lon_c, nbr):
    yi, xi = _nearest_nc(lat_c, lon_c); H, W = nc_lat2d.shape
    if nbr == 0:
        raw = ds_nc["gomofs_u"].isel(y=yi, x=xi).values.astype(np.float64)
        raw[np.abs(raw) > 1e6] = np.nan; ts = (~np.isnan(raw)).astype(float)
    else:
        r0, r1 = max(0, yi - nbr), min(H, yi + nbr + 1)
        c0, c1 = max(0, xi - nbr), min(W, xi + nbr + 1)
        block = ds_nc["gomofs_u"].isel(y=slice(r0, r1), x=slice(c0, c1)).values.astype(np.float64)
        block[np.abs(block) > 1e6] = np.nan; ts = np.mean(~np.isnan(block), axis=(1, 2))
    return nc_times, ts

def plot_full_record(var, loc_data, nbr):
    lbl = VAR_LABEL.get(var, var); fig = go.Figure()
    for slot, loc in loc_data.items():
        times, ts = get_ts(var, loc["lat"], loc["lon"], nbr)
        mean_v = float(np.nanmean(ts)); col = _year_palette(slot, 1)[0]
        fig.add_trace(go.Scatter(x=times, y=ts, mode="lines", line=dict(color=col, width=1.8), name=f"Loc {slot}"))
        fig.add_hline(y=mean_v, line_dash="dash", line_color=col, opacity=0.8, annotation_text=f"{slot} mean: {mean_v:.3f}")
    fig.update_layout(title=f"Full Record: {lbl}", plot_bgcolor="#f0f4f8", paper_bgcolor="#0d1117", font=dict(color="#ddd"), height=320)
    return fig

def plot_year_over_year(var, loc_data, nbr):
    lbl = VAR_LABEL.get(var, var); fig = go.Figure()
    for slot, loc in loc_data.items():
        times, ts = get_ts(var, loc["lat"], loc["lon"], nbr); ts_ser = pd.Series(ts, index=pd.to_datetime(times))
        years = sorted(ts_ser.index.year.unique()); palette = _year_palette(slot, len(years))
        for yi, yr in enumerate(years):
            yd = ts_ser[ts_ser.index.year == yr]; mo = yd.groupby(yd.index.month).mean()
            fig.add_trace(go.Scatter(x=[MONTH_LABELS[m-1] for m in mo.index], y=mo.values, mode="lines+markers", line=dict(color=palette[yi], dash="solid" if slot=="A" else "dash"), name=f"{slot} {yr}"))
    fig.update_layout(title=f"Year-over-Year: {lbl}", plot_bgcolor="#f0f4f8", paper_bgcolor="#0d1117", font=dict(color="#ddd"), height=380)
    return fig

def plot_monthly_boxes(var, loc_data, nbr):
    lbl = VAR_LABEL.get(var, var); fig = go.Figure()
    for slot, loc in loc_data.items():
        times, ts = get_ts(var, loc["lat"], loc["lon"], nbr); ts_ser = pd.Series(ts, index=pd.to_datetime(times))
        col = _year_palette(slot, 1)[0]
        for mo in range(1, 13):
            vals = ts_ser[ts_ser.index.month == mo].values; vals = vals[np.isfinite(vals)]
            fig.add_trace(go.Box(y=vals, name=MONTH_LABELS[mo-1], marker=dict(color=col), fillcolor=col, opacity=0.6, showlegend=(mo==1)))
    fig.update_layout(boxmode="group", title=f"Monthly Dist: {lbl}", plot_bgcolor="#f0f4f8", paper_bgcolor="#0d1117", font=dict(color="#ddd"), height=420)
    return fig

def plot_gomofs_presence(loc_data, nbr):
    fig = go.Figure()
    for slot, loc in loc_data.items():
        times, ts = get_presence_ts(loc["lat"], loc["lon"], nbr); rolling = pd.Series(ts).rolling(30).mean() * 100
        col = _year_palette(slot, 1)[0]
        fig.add_trace(go.Scatter(x=times, y=rolling, mode="lines", line=dict(color=col, width=2), name=f"Loc {slot}"))
    fig.update_layout(title="GOMOFS Data Presence", plot_bgcolor="#f0f4f8", paper_bgcolor="#0d1117", font=dict(color="#ddd"), height=300)
    return fig

def render_ts(_=None):
    var = var_dd.value; nbr = nbr_dd.value; loc_data = {k: v for k, v in click_state.items() if v is not None}
    with out_ts:
        clear_output(wait=True)
        if not loc_data: print("📍 Click the map to select at least one location."); return
        plot_full_record(var, loc_data, nbr).show()
        plot_year_over_year(var, loc_data, nbr).show()
        plot_monthly_boxes(var, loc_data, nbr).show()
        if var in ("gomofs_u", "gomofs_v"): plot_gomofs_presence(loc_data, nbr).show()

def on_plot(_): render_ts()
def on_reset(_):
    click_state["A"] = None; click_state["B"] = None; next_slot[0] = "A"; update_markers()
    status_html.value = "Locations cleared"; with out_ts: clear_output(wait=True)

plot_btn.on_click(on_plot); reset_btn.on_click(on_reset)
nbr_dd.observe(lambda c: render_ts() if any(click_state.values()) else None, names="value")

header = widgets.HTML('<div style="background:linear-gradient(135deg,#0d1117,#161b22); padding:12px; color:#4fc3f7; font-weight:bold;">DOPPIO + GOMOFS Dashboard</div>')
controls = widgets.HBox([var_dd, map_dd, nbr_dd]); buttons = widgets.HBox([plot_btn, reset_btn])

display(header, controls, map_widget, loading_html, status_html, buttons, out_ts)
update_map_data("doppio_temp", "mean")
print("✓ Dashboard ready!")